In [1]:
from pathlib import Path
import time
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from ultralytics import YOLO

from torchvision.models import resnet18
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
)

EXPERIMENTS_DIR = (
    PROJECT_ROOT
    / "experiments"
)

MODELS_DIR = (
    PROJECT_ROOT
    / "models"
)

FINETUNE_DIR = (
    EXPERIMENTS_DIR
    / "finetuning"
)

FINETUNE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)

print("Inspectra — Fine Tuning")
print("=" * 60)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Inspectra — Fine Tuning
Device: 0
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
baseline_models = {
    "bottle": (
        EXPERIMENTS_DIR
        / "bottle"
        / "baseline"
        / "baseline"
        / "weights"
        / "best.pt"
    ),

    "pcb": (
        EXPERIMENTS_DIR
        / "pcb"
        / "baseline"
        / "baseline"
        / "weights"
        / "best.pt"
    ),

    "road": (
        MODELS_DIR
        / "road"
        / "baseline_resnet18.pt"
    ),
}

for name, path in baseline_models.items():

    print(
        f"{name:10}: "
        f"{'FOUND' if path.exists() else 'MISSING'}"
    )

bottle    : FOUND
pcb       : FOUND
road      : FOUND


In [3]:
DETECTION_FINETUNE_CONFIG = {
    "bottle": {
        "epochs": 30,
        "batch": 4,
        "imgsz": 640,
        "lr0": 0.001,
        "lrf": 0.01,
        "weight_decay": 0.0005,
        "mosaic": 1.0,
        "mixup": 0.1,
        "degrees": 5.0,
        "translate": 0.1,
        "scale": 0.5,
        "fliplr": 0.5,
        "flipud": 0.0,
        "hsv_h": 0.015,
        "hsv_s": 0.5,
        "hsv_v": 0.4,
    },

    "pcb": {
        "epochs": 30,
        "batch": 4,
        "imgsz": 640,
        "lr0": 0.001,
        "lrf": 0.01,
        "weight_decay": 0.0005,
        "mosaic": 1.0,
        "mixup": 0.05,
        "degrees": 2.0,
        "translate": 0.05,
        "scale": 0.3,
        "fliplr": 0.5,
        "flipud": 0.0,
        "hsv_h": 0.01,
        "hsv_s": 0.3,
        "hsv_v": 0.3,
    },
}

In [4]:
def finetune_yolo(
    dataset_name,
    config
):

    baseline_path = (
        baseline_models[
            dataset_name
        ]
    )

    if not baseline_path.exists():

        print(
            f"{dataset_name}: "
            "baseline model not found"
        )

        return None

    data_yaml = (
        PROCESSED_DATA
        / dataset_name
        / "data.yaml"
    )

    if not data_yaml.exists():

        print(
            f"{dataset_name}: "
            "data.yaml not found"
        )

        return None

    model = YOLO(
        str(baseline_path)
    )

    output_dir = (
        FINETUNE_DIR
        / dataset_name
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    start = time.perf_counter()

    results = model.train(
        data=str(data_yaml),

        epochs=config["epochs"],

        batch=config["batch"],

        imgsz=config["imgsz"],

        device=DEVICE,

        optimizer="AdamW",

        lr0=config["lr0"],

        lrf=config["lrf"],

        weight_decay=config[
            "weight_decay"
        ],

        mosaic=config["mosaic"],

        mixup=config["mixup"],

        degrees=config["degrees"],

        translate=config["translate"],

        scale=config["scale"],

        fliplr=config["fliplr"],

        flipud=config["flipud"],

        hsv_h=config["hsv_h"],

        hsv_s=config["hsv_s"],

        hsv_v=config["hsv_v"],

        patience=10,

        pretrained=True,

        workers=4,

        seed=42,

        deterministic=True,

        plots=True,

        save=True,

        project=str(
            output_dir
        ),

        name="finetuned",

        exist_ok=True,

        verbose=True,
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    print(
        f"\n{dataset_name.upper()}"
    )

    print(
        f"Fine-tuning time: "
        f"{elapsed / 60:.2f} minutes"
    )

    print(
        "Best model:"
    )

    print(
        output_dir
        / "finetuned"
        / "weights"
        / "best.pt"
    )

    return results

In [5]:
bottle_finetuned = finetune_yolo(
    "bottle",
    DETECTION_FINETUNE_CONFIG[
        "bottle"
    ]
)

New https://pypi.org/project/ultralytics/8.4.118 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.117  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\Inspectra\datasets\processed\bottle\data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr

AMP: checks passed 
WARNING train: Slow image access detected (ping: 0.40.1 ms, read: 4.60.7 MB/s, size: 34.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
train: Scanning D:\Inspectra\datasets\processed\bottle\train\labels.cache... 5420 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5420/5420  0.0s
WARNING val: Slow image access detected (ping: 0.40.1 ms, read: 4.61.5 MB/s, size: 41.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning D:\Inspectra\datasets\processed\bottle\val\labels.cache... 1799 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1799/1799  0.0s
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Plotting labels to D:\Inspectra\experiments\finetuning\bottle\finetuned\labels.jpg... 
Imag

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/30     0.584G     0.5625     0.6161      1.005         19        640: 100% ━━━━━━━━━━━━ 1355/1355 12.6it/s 1:470.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 225/225 13.5it/s 16.7s0.2s
                   all       1799       3479       0.97      0.952      0.982       0.87

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30      0.68G     0.6122     0.6768       1.11         18        640: 0% ──────────── 0/1355  0.1s

...

      28/30      0.68G       0.37     0.2376     0.8705          5        640: 100% ━━━━━━━━━━━━ 1355/1355 15.5it/s 1:270.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 225/225 15.9it/s 14.2s0.1s
                   all       1799       3479      0.969      0.971       0.98      0.899

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      29/30      0.68G     0.4199     0.2696     0.9061          9        640: 0% ──────────── 1/1355 2.2it/s 0.1s<10:28

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr

      29/30      0.68G     0.3612     0.2318     0.8664         11        640: 100% ━━━━━━━━━━━━ 1355/1355 15.5it/s 1:270.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 225/225 16.1it/s 14.0s0.1s
                   all       1799       3479      0.971      0.972      0.981      0.898

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      30/30      0.68G      0.509     0.3023      0.883          4        640: 0% ──────────── 1/1355 2.4it/s 0.1s<9:26

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr

      30/30      0.68G     0.3574     0.2279     0.8638          7        640: 100% ━━━━━━━━━━━━ 1355/1355 15.5it/s 1:270.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 225/225 16.1it/s 14.0s0.1s
                   all       1799       3479      0.968      0.972       0.98      0.901

30 epochs completed in 0.971 hours.
Optimizer stripped from D:\Inspectra\experiments\finetuning\bottle\finetuned\weights\last.pt, 6.2MB
Optimizer stripped from D:\Inspectra\experiments\finetuning\bottle\finetuned\weights\best.pt, 6.2MB

Validating D:\Inspectra\experiments\finetuning\bottle\finetuned\weights\best.pt...
Ultralytics 8.4.117  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 225/225 16.7it/s 13.5s0.1s
    

In [6]:
pcb_finetuned = finetune_yolo(
    "pcb",
    DETECTION_FINETUNE_CONFIG[
        "pcb"
    ]
)

New https://pypi.org/project/ultralytics/8.4.118 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.117  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=d:\Inspectra\datasets\processed\pcb\data.yaml, degrees=2.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.01, hsv_s=0.3, hsv_v=0.3, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_rat

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr

AMP: checks passed 
WARNING train: Slow image access detected (ping: 0.40.2 ms, read: 15.36.3 MB/s, size: 94.4 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
train: Scanning D:\Inspectra\datasets\processed\pcb\train\labels.cache... 6370 images, 2164 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 8534/8534  0.0s
WARNING val: Slow image access detected (ping: 0.50.2 ms, read: 15.15.3 MB/s, size: 105.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning D:\Inspectra\datasets\processed\pcb\val\labels.cache... 802 images, 264 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1066/1066  0.0s
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Plotting labels to D:\Inspectra\experiments\finetuning\pcb\finetuned\labels.jpg... 
Image 

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr

       1/30     0.629G       1.66       1.02      1.146          1        640: 100% ━━━━━━━━━━━━ 2134/2134 15.4it/s 2:190.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 134/134 16.3it/s 8.2s0.1s
                   all       1066       1595      0.968      0.959      0.979      0.515

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30     0.725G      1.792      0.946       1.16         10        640: 0% ──────────── 1/2134 1.9it/s 0.2s<18:20

...

      28/30     0.725G      1.352     0.6332      1.031          3        640: 100% ━━━━━━━━━━━━ 2134/2134 15.7it/s 2:160.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 134/134 16.3it/s 8.2s0.1s
                   all       1066       1595      0.981      0.982      0.989      0.566

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      29/30     0.725G      1.576     0.7215      1.038          8        640: 0% ──────────── 1/2134 2.2it/s 0.1s<16:04

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr

      29/30     0.725G      1.328     0.6026      1.028          2        640: 100% ━━━━━━━━━━━━ 2134/2134 15.7it/s 2:160.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 134/134 16.6it/s 8.1s0.1s
                   all       1066       1595      0.981      0.981      0.987       0.57

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      30/30     0.725G      1.295     0.7099      1.033          6        640: 0% ──────────── 1/2134 2.3it/s 0.1s<15:43

c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\Context.cpp:95.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
c:\Users\Garvit\AppData\Local\Programs\Python\Python311\Lib\site-packages\ultralytics\utils\tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Tr

      30/30     0.725G      1.317     0.6378      1.021          3        640: 100% ━━━━━━━━━━━━ 2134/2134 15.9it/s 2:140.1ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 134/134 16.4it/s 8.1s0.1s
                   all       1066       1595      0.982      0.982      0.989      0.569

30 epochs completed in 1.204 hours.
Optimizer stripped from D:\Inspectra\experiments\finetuning\pcb\finetuned\weights\last.pt, 6.2MB
Optimizer stripped from D:\Inspectra\experiments\finetuning\pcb\finetuned\weights\best.pt, 6.2MB

Validating D:\Inspectra\experiments\finetuning\pcb\finetuned\weights\best.pt...
Ultralytics 8.4.117  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 134/134 17.2it/s 7.8s0.1s
               

In [7]:
ROAD_ROOT = (
    PROCESSED_DATA
    / "road"
)

ROAD_IMAGE_SIZE = 224

road_train_transform = transforms.Compose([
    transforms.Resize(
        (ROAD_IMAGE_SIZE, ROAD_IMAGE_SIZE)
    ),
    transforms.RandomHorizontalFlip(
        p=0.5
    ),
    transforms.RandomRotation(
        degrees=5
    ),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
])

road_eval_transform = transforms.Compose([
    transforms.Resize(
        (ROAD_IMAGE_SIZE, ROAD_IMAGE_SIZE)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
])

In [8]:
road_train = datasets.ImageFolder(
    ROAD_ROOT / "train",
    transform=road_train_transform
)

road_val = datasets.ImageFolder(
    ROAD_ROOT / "val",
    transform=road_eval_transform
)

road_train_loader = DataLoader(
    road_train,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

road_val_loader = DataLoader(
    road_val,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

print(
    "Classes:",
    road_train.classes
)

print(
    "Train:",
    len(road_train)
)

print(
    "Validation:",
    len(road_val)
)

Classes: ['Negative', 'Positive']
Train: 28000
Validation: 6000


In [9]:
road_device = (
    f"cuda:{DEVICE}"
    if isinstance(DEVICE, int)
    else DEVICE
)

road_model = resnet18(
    weights=None
)

road_model.fc = torch.nn.Linear(
    road_model.fc.in_features,
    len(road_train.classes)
)

road_state = torch.load(
    baseline_models["road"],
    map_location=road_device
)

road_model.load_state_dict(
    road_state
)

road_model = road_model.to(
    road_device
)

print(
    "Baseline Road model loaded"
)

Baseline Road model loaded


C:\Users\Garvit\AppData\Local\Temp\ipykernel_13824\3694774701.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  road_state = torch.load(


In [10]:
road_criterion = (
    torch.nn.CrossEntropyLoss()
)

road_optimizer = torch.optim.AdamW(
    road_model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

road_scheduler = (
    torch.optim.lr_scheduler.CosineAnnealingLR(
        road_optimizer,
        T_max=10
    )
)

In [11]:
def train_road_epoch():

    road_model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in road_train_loader:

        images = images.to(
            road_device
        )

        labels = labels.to(
            road_device
        )

        road_optimizer.zero_grad()

        outputs = road_model(
            images
        )

        loss = road_criterion(
            outputs,
            labels
        )

        loss.backward()

        road_optimizer.step()

        total_loss += (
            loss.item()
            * images.size(0)
        )

        predictions = (
            outputs.argmax(
                dim=1
            )
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    return (
        total_loss / total,
        correct / total
    )

In [12]:
@torch.no_grad()
def validate_road():

    road_model.eval()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in road_val_loader:

        images = images.to(
            road_device
        )

        labels = labels.to(
            road_device
        )

        outputs = road_model(
            images
        )

        loss = road_criterion(
            outputs,
            labels
        )

        total_loss += (
            loss.item()
            * images.size(0)
        )

        predictions = (
            outputs.argmax(
                dim=1
            )
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    return (
        total_loss / total,
        correct / total
    )

In [13]:
ROAD_FINETUNE_EPOCHS = 10

road_finetune_history = []

best_val_accuracy = 0.0

road_finetune_dir = (
    MODELS_DIR
    / "road"
    / "finetuned"
)

road_finetune_dir.mkdir(
    parents=True,
    exist_ok=True
)

for epoch in range(
    ROAD_FINETUNE_EPOCHS
):

    train_loss, train_accuracy = (
        train_road_epoch()
    )

    val_loss, val_accuracy = (
        validate_road()
    )

    road_scheduler.step()

    road_finetune_history.append(
        {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
        }
    )

    print(
        f"Epoch {epoch + 1:02d}/"
        f"{ROAD_FINETUNE_EPOCHS} "
        f"| train_loss={train_loss:.6f} "
        f"| train_acc={train_accuracy:.5f} "
        f"| val_loss={val_loss:.6f} "
        f"| val_acc={val_accuracy:.5f}"
    )

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = (
            val_accuracy
        )

        torch.save(
            road_model.state_dict(),
            road_finetune_dir
            / "resnet18_finetuned.pt"
        )

Epoch 01/10 | train_loss=0.007191 | train_acc=0.99771 | val_loss=0.003195 | val_acc=0.99933
Epoch 02/10 | train_loss=0.002201 | train_acc=0.99946 | val_loss=0.002881 | val_acc=0.99933
Epoch 03/10 | train_loss=0.002037 | train_acc=0.99950 | val_loss=0.002507 | val_acc=0.99950
Epoch 04/10 | train_loss=0.001562 | train_acc=0.99957 | val_loss=0.002776 | val_acc=0.99933
Epoch 05/10 | train_loss=0.001330 | train_acc=0.99961 | val_loss=0.002334 | val_acc=0.99933
Epoch 06/10 | train_loss=0.001176 | train_acc=0.99971 | val_loss=0.002083 | val_acc=0.99950
Epoch 07/10 | train_loss=0.000843 | train_acc=0.99986 | val_loss=0.002547 | val_acc=0.99950
Epoch 08/10 | train_loss=0.000791 | train_acc=0.99986 | val_loss=0.002156 | val_acc=0.99950
Epoch 09/10 | train_loss=0.000631 | train_acc=0.99982 | val_loss=0.002291 | val_acc=0.99950
Epoch 10/10 | train_loss=0.000612 | train_acc=0.99989 | val_loss=0.002391 | val_acc=0.99950


In [14]:
road_finetune_df = pd.DataFrame(
    road_finetune_history
)

plt.figure(
    figsize=(10, 5)
)

plt.plot(
    road_finetune_df["epoch"],
    road_finetune_df["train_accuracy"],
    label="Train Accuracy"
)

plt.plot(
    road_finetune_df["epoch"],
    road_finetune_df["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Accuracy"
)

plt.title(
    "Road Fine-Tuning"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.show()

<Figure size 1000x500 with 1 Axes>

In [15]:
finetune_metadata = {
    "bottle": DETECTION_FINETUNE_CONFIG[
        "bottle"
    ],

    "pcb": DETECTION_FINETUNE_CONFIG[
        "pcb"
    ],

    "road": {
        "epochs": ROAD_FINETUNE_EPOCHS,
        "optimizer": "AdamW",
        "learning_rate": 1e-5,
        "weight_decay": 1e-4,
        "scheduler": "CosineAnnealingLR",
        "best_validation_accuracy": best_val_accuracy,
    }
}

with open(
    FINETUNE_DIR
    / "experiment_config.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        finetune_metadata,
        file,
        indent=4
    )

print(
    "Fine-tuning metadata saved."
)

Fine-tuning metadata saved.
